<a href="https://colab.research.google.com/github/sascha-pst/McCombs-DSBA/blob/main/FoodHubProject_Git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FoodHub Order Analysis: Exploratory Insights & Recommendations

> An exploratory data analysis of 1,898 food delivery orders from FoodHub, a New York food aggregator app. Using pandas, NumPy, matplotlib, and seaborn, I examine where demand is concentrated, how fast orders reach customers, what ratings reveal (and hide), and where revenue comes from, then translate those findings into business recommendations.

## Executive Summary

| # | Finding | Evidence | So what |
|---|---|---|---|
| 1 | **Demand is concentrated** | 4 cuisines (American, Japanese, Italian, Chinese) make up **83%** of orders; Shake Shack alone is **11.5%** | Partnerships with a few restaurants move the needle, but create dependency risk |
| 2 | **Weekday deliveries are slower** | Weekday mean delivery **28.3 min** vs weekend **22.5 min** (~6 min, ~21% slower) | Weekday driver supply is the clearest operational lever |
| 3 | **Customers who rate are satisfied, but most don't rate** | **39%** of orders unrated; among rated orders ~**51%** are 5-star and **none are below 3** | The feedback loop is weak, not negative |
| 4 | **Service is mostly fast** | Only **10.5%** of orders exceed 60 min from placement to delivery | Speed is a credible marketing message |
| 5 | **Higher-value orders drive revenue** | **29%** of orders exceed \$20 and earn the higher 25% commission, generating **60%** of revenue | Nudging baskets over \$20 is a direct revenue lever |

Details, charts, and recommendations follow below.

## Business Context

FoodHub connects students and professionals with a wide range of New York restaurants. The company wants to use its order data to understand restaurant demand, improve customer experience, and strengthen operations in a competitive delivery market. FoodHub earns a commission on each order: **25%** on orders above \$20, **15%** on orders above \$5, and nothing on orders of \$5 or less.

### Data Dictionary

| Column | Description |
|---|---|
| `order_id` | Unique order identifier |
| `customer_id` | Customer identifier |
| `restaurant_name` | Restaurant name |
| `cuisine_type` | Type of cuisine ordered |
| `cost_of_the_order` | Order cost (\$) |
| `day_of_the_week` | Weekday (Mon–Fri) or Weekend (Sat–Sun) |
| `rating` | Customer rating out of 5, or "Not given" |
| `food_preparation_time` | Restaurant prep time (minutes) |
| `delivery_time` | Time from pickup to drop-off (minutes) |

## 1. Setup and Data Loading

In [ ]:
# Pinned versions for reproducibility (Colab)
!pip install numpy==1.25.2 pandas==1.5.3 matplotlib==3.7.1 seaborn==0.13.1 -q --user

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Consistent styling for every chart in the report
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False})
BASE = "#9BB0C8"      # neutral bars
ACCENT = "#1F4E79"    # highlighted bars / key series
pd.set_option("display.max_columns", None)

In [ ]:
# Load the data ONCE; every later cell works from this dataframe
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/foodhub_order.csv')
df.head()

## 2. Data Overview and Preparation

In [ ]:
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
print(f"Restaurants: {df['restaurant_name'].nunique()}   Cuisines: {df['cuisine_type'].nunique()}   "
      f"Customers: {df['customer_id'].nunique():,}")
df.dtypes

In [ ]:
df.describe().round(2)

In [ ]:
# The only 'missing' data is ratings recorded as the string 'Not given'
not_rated = (df['rating'] == 'Not given')
print(f"Unrated orders: {not_rated.sum():,} ({not_rated.mean():.1%} of all orders)")
print(f"Any true nulls in the file: {df.isnull().sum().sum()}")

**Preparation choices.** Rather than filling the 39% of missing ratings (imputing the mode would inflate 5-star ratings and pull every correlation toward zero), I keep them as missing and analyse ratings on rated orders only. I also add two derived columns: a numeric rating and the total time from order placement to delivery.

In [ ]:
df['is_rated'] = ~not_rated
df['rating_num'] = pd.to_numeric(df['rating'], errors='coerce')        # 'Not given' -> NaN
df['total_time'] = df['food_preparation_time'] + df['delivery_time']
df[['cost_of_the_order', 'food_preparation_time', 'delivery_time', 'total_time']].describe().round(1)

## 3. Where Is Demand Concentrated?

In [ ]:
cuisine = df['cuisine_type'].value_counts()
share = cuisine / len(df)
top4_share = share.head(4).sum()

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [ACCENT if i < 4 else BASE for i in range(len(cuisine))]
ax.barh(cuisine.index, cuisine.values, color=colors)
ax.invert_yaxis()
for y, (n, p) in enumerate(zip(cuisine.values, share.values)):
    ax.text(n + 5, y, f"{p:.0%}", va='center', fontsize=9)
ax.set_xlim(0, cuisine.max() * 1.1)
ax.set_title(f"Four cuisines account for {top4_share:.0%} of all orders")
ax.set_xlabel("Number of orders"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
top_rest = df['restaurant_name'].value_counts().head(10)
top_rest_share = top_rest / len(df)

fig, ax = plt.subplots(figsize=(9, 5))
colors = [ACCENT] + [BASE] * (len(top_rest) - 1)
ax.barh(top_rest.index, top_rest.values, color=colors)
ax.invert_yaxis()
for y, (n, p) in enumerate(zip(top_rest.values, top_rest_share.values)):
    ax.text(n + 2, y, f"{n} ({p:.1%})", va='center', fontsize=9)
ax.set_xlim(0, top_rest.max() * 1.18)
ax.set_title(f"{top_rest.index[0]} alone receives {top_rest_share.iloc[0]:.1%} of all orders "
             f"(top 10 of {df['restaurant_name'].nunique()} restaurants)")
ax.set_xlabel("Number of orders"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
day_counts = df['day_of_the_week'].value_counts()
weekend_share = day_counts['Weekend'] / len(df)
weekend_top = df.loc[df['day_of_the_week'] == 'Weekend', 'cuisine_type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh(['Orders'], [day_counts['Weekend']], color=ACCENT, label='Weekend')
ax.barh(['Orders'], [day_counts['Weekday']], left=[day_counts['Weekend']], color=BASE, label='Weekday')
ax.text(day_counts['Weekend'] / 2, 0, f"Weekend\n{weekend_share:.0%}", ha='center', va='center', color='white', fontweight='bold')
ax.text(day_counts['Weekend'] + day_counts['Weekday'] / 2, 0, f"Weekday\n{1 - weekend_share:.0%}", ha='center', va='center')
ax.set_title(f"{weekend_share:.0%} of orders are placed on weekends (2 of 7 days)")
ax.set_xlabel("Number of orders"); ax.set_yticks([])
plt.tight_layout(); plt.show()

print(f"Most popular weekend cuisine: {weekend_top.index[0]} ({weekend_top.iloc[0]} orders)")

**Takeaway.** Demand is heavily concentrated: a handful of cuisines and restaurants carry most of the volume, and weekends carry most of the week. This makes targeted restaurant partnerships high-leverage, but it also means losing one top partner would be felt immediately.

## 4. How Fast Do Orders Arrive?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['food_preparation_time'], discrete=True, color=BASE, ax=axes[0])
axes[0].set_title("Prep time is spread evenly from 20 to 35 min")
axes[0].set_xlabel("Food preparation time (minutes)"); axes[0].set_ylabel("Orders")
sns.histplot(df['delivery_time'], discrete=True, color=BASE, ax=axes[1])
axes[1].set_title("Delivery time clusters in two groups")
axes[1].set_xlabel("Delivery time (minutes)"); axes[1].set_ylabel("Orders")
plt.tight_layout(); plt.show()

The two humps in delivery time suggest two different populations of orders. Splitting by day of week explains them:

In [ ]:
means = df.groupby('day_of_the_week')['delivery_time'].agg(['mean', 'std', 'count']).round(2)
gap = means.loc['Weekday', 'mean'] - means.loc['Weekend', 'mean']

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='day_of_the_week', y='delivery_time', order=['Weekday', 'Weekend'],
            hue='day_of_the_week', palette={'Weekday': ACCENT, 'Weekend': BASE},
            legend=False, width=0.5, ax=ax)
for x, day in enumerate(['Weekday', 'Weekend']):
    m = means.loc[day, 'mean']
    ax.scatter(x, m, color='white', edgecolor='black', zorder=3, s=50)
    ax.text(x + 0.3, m, f"mean {m:.1f} min", va='center', fontsize=10)
ax.set_title(f"Weekday deliveries take ~{gap:.0f} minutes longer than weekend deliveries")
ax.set_xlabel(""); ax.set_ylabel("Delivery time (minutes)")
plt.tight_layout(); plt.show()

means.rename(columns={'mean': 'mean_min', 'std': 'std_min', 'count': 'orders'})

In [ ]:
over_60 = (df['total_time'] > 60).mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(df['total_time'], discrete=True, color=BASE, ax=ax)
ax.axvline(60, color='firebrick', linestyle='--')
ax.text(60.5, ax.get_ylim()[1] * 0.9, f"{over_60:.1%} of orders\ntake > 60 min", color='firebrick')
ax.set_title(f"Most orders arrive within an hour (mean {df['total_time'].mean():.0f} min from order to door)")
ax.set_xlabel("Total time: preparation + delivery (minutes)"); ax.set_ylabel("Orders")
plt.tight_layout(); plt.show()

**Takeaway.** Nearly 9 in 10 orders arrive within an hour, which supports speed as a selling point. The bigger opportunity is the weekday gap: weekdays have far fewer orders yet slower deliveries, which points to driver supply (not restaurant prep) as the likely cause, since prep time is spread evenly across the week. This is a hypothesis to test, since the data does not include driver counts or distance.

## 5. What Do Ratings Tell Us?

In [ ]:
rated = df[df['is_rated']].copy()
rating_share = rated['rating_num'].value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={'width_ratios': [1, 1.6]})

axes[0].pie([df['is_rated'].sum(), (~df['is_rated']).sum()], labels=['Rated', 'Not rated'],
            colors=[ACCENT, BASE], autopct='%1.0f%%', startangle=90,
            wedgeprops={'width': 0.4}, pctdistance=0.8, textprops={'fontsize': 10})
for t in axes[0].texts[1::2]:
    t.set_color('white'); t.set_fontweight('bold')
axes[0].set_title(f"{(~df['is_rated']).mean():.0%} of orders go unrated")

axes[1].bar(rating_share.index.astype(str), rating_share.values,
            color=[ACCENT if r == 5 else BASE for r in rating_share.index])
for x, p in enumerate(rating_share.values):
    axes[1].text(x, p + 0.01, f"{p:.0%}", ha='center')
axes[1].set_ylim(0, rating_share.max() * 1.15)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_title("Among rated orders, half are 5 stars and none are below 3")
axes[1].set_xlabel("Rating"); axes[1].set_ylabel("Share of rated orders")
plt.tight_layout(); plt.show()

In [ ]:
# Do ratings move with speed or cost? Compare averages by rating (rated orders only, no imputation)
by_rating = rated.groupby('rating_num')[['delivery_time', 'food_preparation_time', 'total_time', 'cost_of_the_order']].mean().round(1)
by_rating['orders'] = rated['rating_num'].value_counts().sort_index()
display(by_rating)

# Rated vs unrated: is non-rating a hidden signal of a bad experience?
display(df.groupby('is_rated')[['delivery_time', 'total_time', 'cost_of_the_order']].mean().round(1)
          .rename(index={True: 'Rated', False: 'Not rated'}))

print("Correlation of rating with each variable (rated orders only):")
print(rated[['rating_num', 'delivery_time', 'food_preparation_time', 'total_time', 'cost_of_the_order']]
      .corr()['rating_num'].drop('rating_num').round(3).to_string())

**Takeaway.** Customers who leave ratings are overwhelmingly satisfied, and unrated orders look the same as rated ones: both average about 24 minutes for delivery and \$16–17 per order. So silence doesn't seem to mean a bad experience.

Ratings also barely move with speed or price. 3-, 4- and 5-star orders all average about 24 minutes for delivery and 27 minutes for prep, and every correlation is close to zero. Whatever separates a 3-star order from a 5-star one (food quality, accuracy, packaging) isn't in this data. That is a finding in itself: FoodHub's feedback loop is too thin to guide decisions.

## 6. Where Does Revenue Come From?

In [ ]:
cost = df['cost_of_the_order']
# Tiered commission: 25% on orders > $20, 15% on orders > $5, 0% otherwise
df['revenue'] = np.select([cost > 20, cost > 5], [cost * 0.25, cost * 0.15], default=0)
df['tier'] = np.select([cost > 20, cost > 5], ['> $20 (25%)', '$5–20 (15%)'], default='≤ $5 (0%)')

total_revenue = df['revenue'].sum()
tier_summary = df.groupby('tier').agg(orders=('revenue', 'size'), revenue=('revenue', 'sum'))
tier_summary['share_of_orders'] = (tier_summary['orders'] / len(df)).round(3)
tier_summary['share_of_revenue'] = (tier_summary['revenue'] / total_revenue).round(3)

print(f"Total FoodHub revenue across {len(df):,} orders: ${total_revenue:,.2f}")
print(f"Average revenue per order: ${total_revenue / len(df):.2f}")
tier_summary.round(2)

In [ ]:
tiers = ['$5–20 (15%)', '> $20 (25%)']
ts = tier_summary.reindex(tiers).fillna(0)

fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(2)
ax.barh(x + 0.2, ts['share_of_orders'], height=0.4, color=BASE, label='Share of orders')
ax.barh(x - 0.2, ts['share_of_revenue'], height=0.4, color=ACCENT, label='Share of revenue')
for i, t in enumerate(tiers):
    ax.text(ts.loc[t, 'share_of_orders'] + 0.01, i + 0.2, f"{ts.loc[t, 'share_of_orders']:.0%}", va='center')
    ax.text(ts.loc[t, 'share_of_revenue'] + 0.01, i - 0.2, f"{ts.loc[t, 'share_of_revenue']:.0%}", va='center')
ax.set_yticks(x); ax.set_yticklabels(tiers)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.set_xlim(0, 1)
hi = ts.loc['> $20 (25%)']
ax.set_title(f"Orders over $20 are {hi['share_of_orders']:.0%} of volume but {hi['share_of_revenue']:.0%} of revenue")
ax.legend(loc='lower right', frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
order = df.groupby('cuisine_type')['cost_of_the_order'].median().sort_values(ascending=False).index
counts = df['cuisine_type'].value_counts()

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=df, y='cuisine_type', x='cost_of_the_order', order=order, color=BASE, ax=ax)
ax.axvline(20, color='firebrick', linestyle='--')
ax.set_yticks(range(len(order)), [f"{c} (n={counts[c]})" for c in order])
ax.set_title("Pricier cuisines (e.g. French, Thai) are among the least ordered")
ax.set_xlabel("Order cost ($)   |   red dashed line = $20 commission threshold"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

**Takeaway.** The typical order sits below the \$20 threshold, yet orders above it produce a disproportionate share of revenue. Small nudges that lift baskets over \$20 (add-on suggestions, bundle deals, free-delivery minimums) would shift orders into the higher commission tier without needing new customers.

## 7. Promotion Eligibility

FoodHub wants to promote restaurants with **more than 50 ratings** and an **average rating above 4**.

In [ ]:
rest_ratings = rated.groupby('restaurant_name')['rating_num'].agg(rating_count='count', avg_rating='mean')
eligible = (rest_ratings[(rest_ratings['rating_count'] > 50) & (rest_ratings['avg_rating'] > 4)]
            .sort_values('avg_rating', ascending=False).round(2))
print(f"{len(eligible)} restaurants qualify for the promotion:")
eligible

## 8. Quick Answers to Key Business Questions

In [ ]:
top_customers = df['customer_id'].value_counts().head(3)
answers = pd.Series({
    "Top 5 restaurants by orders": ", ".join(df['restaurant_name'].value_counts().head(5).index),
    "Most popular weekend cuisine": f"{weekend_top.index[0]} ({weekend_top.iloc[0]} orders)",
    "Orders costing more than $20": f"{(cost > 20).mean():.1%}",
    "Mean delivery time": f"{df['delivery_time'].mean():.1f} min",
    "Mean total time (prep + delivery)": f"{df['total_time'].mean():.1f} min",
    "Orders taking over 60 min in total": f"{over_60:.1%}",
    "Top 3 customers (orders)": ", ".join(f"{c} ({n})" for c, n in top_customers.items()),
    "Top 3 customers' share of orders": f"{top_customers.sum() / len(df):.1%}",
    "Total FoodHub revenue": f"${total_revenue:,.2f}",
    "Restaurants eligible for promotion": ", ".join(eligible.index),
}, name="Answer")
pd.set_option("display.max_colwidth", None)
answers.to_frame()

## 9. Conclusions and Recommendations

**1. Protect and deepen top restaurant partnerships.** Four cuisines drive 83% of orders and the top five restaurants alone account for about a third. Prioritise these partners (featured placement, co-marketing, the promotion above), while recruiting more restaurants in under-served cuisines to reduce dependency on a few names.

**2. Close the weekday delivery gap.** Weekday deliveries average about 6 minutes longer than weekend deliveries even though weekdays have far less volume. A first step is to check driver availability and routing on weekdays; if the gap is a staffing issue, a modest weekday incentive for drivers could bring weekday service in line with weekends.

**3. Lift more orders over \$20.** Orders above \$20 are 29% of volume but 60% of revenue. Bundle suggestions, add-on prompts at checkout, or a free-delivery threshold just above \$20 would move orders into the higher commission tier.

**4. Rebuild the feedback loop.** 39% of orders go unrated and the ratings that exist are clustered at 4–5 stars, so they cannot distinguish good from great or explain what drives satisfaction. A one-tap post-delivery prompt, plus a short optional question on what went well or badly, would give FoodHub data it can act on.

**5. Market on speed and choice.** With nearly 90% of orders arriving within an hour and 178 restaurants across 14 cuisines, speed and variety are credible value propositions for customer acquisition.

### Limitations and Next Steps

- The data covers a single snapshot with no dates, so trends and seasonality can't be assessed.
- There is no information on delivery distance, driver counts, or customer acquisition cost, so the drivers of the weekday gap and true profitability remain hypotheses.
- Ratings only span 3–5 and are missing for 39% of orders, which limits any analysis of satisfaction.
- **Natural extension:** a statistical test of the weekday/weekend delivery difference, and a model predicting delivery time from restaurant, cuisine, and day, would turn these observations into quantified, testable drivers.